# A3 (часть 2). Transfer learning, fine-tuning и LoRA для классификации текстов

> **Запускать в Google Colab с GPU:** *Runtime → Change runtime type → T4 GPU*. Время ≈ 5–10 минут.
> На CPU тоже работает, но медленно (≈ 30–60 минут).
> Результаты сохраняются в `reports/transformer_results.json`, их подхватывает сводная таблица в `03_model_training.ipynb`.

**Задача:** тот же F-01 — тип происшествия по тексту, **то же разбиение** (`src/nlp.split_texts`, seed 42) и **те же метрики** (accuracy, macro-F1), что у классических моделей.

| Эксперимент | Что обучается | Идея |
|---|---|---|
| 1. Transfer learning (feature extraction) | только логистическая регрессия поверх замороженного энкодера | берём готовые «знания языка» модели как признаки |
| 2. Full fine-tuning | **все** веса трансформера + классификатор | дообучаем модель целиком под нашу задачу |
| 3. LoRA (PEFT) | маленькие матрицы-адаптеры A·B в слоях внимания + классификатор | дообучаем ≈ 1–5% параметров, остальное заморожено |

**Исходная модель — `cointegrated/rubert-tiny2`.**
- Язык: описания на **русском**. Модель предобучена на русских текстах, а многоязычные модели тратят ёмкость на десятки языков.
- Размер: ≈ 29 млн параметров. Помещается в бесплатный Colab и обучается за минуты.
- Предметная область: модель общего назначения, но описания написаны обычным языком («упал», «госпитализирован», «пострадавших нет»). Специальная HSE-модель не нужна: значение этих слов модель уже знает.

In [1]:
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers", "peft"], check=True)
    REPO_URL = ""   # ← ссылка на ваш GitHub-репозиторий (или оставьте пустым и загрузите zip)
    if not list(Path(".").rglob("src/data.py")):
        if REPO_URL:
            subprocess.run(["git", "clone", REPO_URL, "HSE_AI_Essentials"], check=True)
        else:
            from google.colab import files
            print("Загрузите zip-архив папки проекта")
            uploaded = files.upload()
            subprocess.run(["unzip", "-q", "-o", next(iter(uploaded))], check=True)
    ROOT = next(Path(".").rglob("src/data.py")).parent.parent.resolve()   # папка, где лежит src/
else:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
print("Проект:", ROOT)

Проект: c:\Users\Admin\PY Projects\AITU\HSE_AI_Essentials


In [2]:
import json, math, random, time
import numpy as np
import pandas as pd
import torch
import transformers, peft, sklearn
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from peft import LoraConfig, get_peft_model, TaskType
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report

from src.data import load_clean, INCIDENT_TYPES
from src.nlp import split_texts, normalize

SEED = 42
def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
set_seed()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_NAME = "cointegrated/rubert-tiny2"
MAX_LEN = 128
print(f"device={device} | torch {torch.__version__} | transformers {transformers.__version__} | "
      f"peft {peft.__version__} | sklearn {sklearn.__version__}")

c:\Users\Admin\PY Projects\AITU\HSE_AI_Essentials\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device=cpu | torch 2.15.0+cpu | transformers 5.22.2 | peft 0.21.2 | sklearn 1.9.1


## Данные: то же разбиение, что у классических моделей

In [3]:
incidents, _ = load_clean(ROOT / "data")
train_df, val_df, test_df = split_texts(incidents)
label2id = {t: i for i, t in enumerate(INCIDENT_TYPES)}
id2label = {i: t for t, i in label2id.items()}

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
def encode(df):
    enc = tokenizer([normalize(t) for t in df["description"]], truncation=True,
                    max_length=MAX_LEN, padding="max_length", return_tensors="pt")
    labels = torch.tensor(df["type"].map(label2id).values)
    return TensorDataset(enc["input_ids"], enc["attention_mask"], labels)

train_ds, val_ds, test_ds = encode(train_df), encode(val_df), encode(test_df)
lengths = [len(tokenizer.tokenize(normalize(t))) for t in train_df["description"]]
print({"train": len(train_ds), "val": len(val_ds), "test": len(test_ds)},
      "| токенов в описании: медиана", int(np.median(lengths)), "максимум", max(lengths))

# Веса классов для функции потерь: редкий класс (авария) весит больше
counts = train_df["type"].value_counts().reindex(INCIDENT_TYPES).values
class_weights = torch.tensor(len(train_df) / (len(INCIDENT_TYPES) * counts), dtype=torch.float)
print("Веса классов:", dict(zip(INCIDENT_TYPES, class_weights.numpy().round(2))))

c:\Users\Admin\PY Projects\AITU\HSE_AI_Essentials\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Admin\.cache\huggingface\hub\models--cointegrated--rubert-tiny2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


{'train': 1040, 'val': 223, 'test': 224} | токенов в описании: медиана 38 максимум 59
Веса классов: {'Опасная ситуация': np.float32(0.37), 'Микротравма': np.float32(0.91), 'Несчастный случай': np.float32(6.5), 'Авария': np.float32(21.67)}


## Эксперимент 1. Transfer learning: замороженный энкодер как генератор признаков

Каждое описание пропускаем через предобученную модель **без обучения** и усредняем векторы токенов (mean pooling). Получаем 312-мерный «смысловой» вектор текста. Поверх него обучаем логистическую регрессию.

**Какие знания переносятся:** понимание русского языка — синонимы, словоформы («госпитализирован» ≈ «доставлен в больницу»), контекст. Этого нет у TF-IDF: для него «перелом» и «переломом» — разные слова.

In [4]:
@torch.no_grad()
def embed(ds, encoder, batch_size=64):
    encoder.eval(); out = []
    for ids, mask, _ in DataLoader(ds, batch_size=batch_size):
        hidden = encoder(input_ids=ids.to(device), attention_mask=mask.to(device)).last_hidden_state
        m = mask.to(device).unsqueeze(-1).float()
        out.append(((hidden * m).sum(1) / m.sum(1)).cpu().numpy())
    return np.vstack(out)

t0 = time.time()
encoder = AutoModel.from_pretrained(MODEL_NAME).to(device)
X_train, X_val, X_test = embed(train_ds, encoder), embed(val_ds, encoder), embed(test_ds, encoder)
probe = LogisticRegression(max_iter=5000, class_weight="balanced", C=10, random_state=SEED)
probe.fit(X_train, train_df["type"])
tl_time = time.time() - t0

def metrics(y_true, y_pred):
    return {"accuracy": round(accuracy_score(y_true, y_pred), 4),
            "macro_f1": round(f1_score(y_true, y_pred, average="macro"), 4)}

results = {}
results["Transfer learning (frozen rubert-tiny2 + LogReg)"] = {
    "val": metrics(val_df["type"], probe.predict(X_val)),
    "test": metrics(test_df["type"], probe.predict(X_test)),
    "trainable_params": int(probe.coef_.size + probe.intercept_.size),
    "train_time_sec": round(tl_time, 1),
    "params": {"pooling": "mean", "classifier": "LogisticRegression(C=10, class_weight=balanced)"},
}
results["Transfer learning (frozen rubert-tiny2 + LogReg)"]

c:\Users\Admin\PY Projects\AITU\HSE_AI_Essentials\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Admin\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 55/55 [00:00<00:00, 7601.13it/s]
[transformers] BertModel LOAD REPORT from: cointegrated

{'val': {'accuracy': 0.9283, 'macro_f1': 0.8909},
 'test': {'accuracy': 0.9107, 'macro_f1': 0.8858},
 'trainable_params': 1252,
 'train_time_sec': 41.2,
 'params': {'pooling': 'mean',
  'classifier': 'LogisticRegression(C=10, class_weight=balanced)'}}

## Общий цикл обучения (эксперименты 2 и 3)

- **Loss:** взвешенная кросс-энтропия (веса классов выше), чтобы модель не игнорировала аварии.
- **Optimizer:** AdamW (Adam с правильной регуляризацией весов), linear warmup 10% + затухание learning rate.
- **Выбор эпохи:** после каждой эпохи считаем macro-F1 на validation и сохраняем лучшее состояние (early stopping по лучшему результату). Test в обучении не участвует.

In [5]:
@torch.no_grad()
def predict(model, ds, batch_size=64):
    model.eval(); preds = []
    for ids, mask, _ in DataLoader(ds, batch_size=batch_size):
        logits = model(input_ids=ids.to(device), attention_mask=mask.to(device)).logits
        preds.append(logits.argmax(-1).cpu())
    return [id2label[i] for i in torch.cat(preds).tolist()]

def train(model, lr, epochs, batch_size=16, weight_decay=0.01):
    set_seed()
    model.to(device)
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(params, lr=lr, weight_decay=weight_decay)
    loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                        generator=torch.Generator().manual_seed(SEED))
    total = epochs * len(loader)
    scheduler = get_linear_schedule_with_warmup(optimizer, int(0.1 * total), total)
    loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights.to(device))

    best_f1, best_state, history = -1.0, None, []
    t0 = time.time()
    for epoch in range(1, epochs + 1):
        model.train(); running = 0.0
        for ids, mask, labels in loader:
            optimizer.zero_grad()
            logits = model(input_ids=ids.to(device), attention_mask=mask.to(device)).logits
            loss = loss_fn(logits, labels.to(device))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            optimizer.step(); scheduler.step()
            running += loss.item()
        val_f1 = f1_score(val_df["type"], predict(model, val_ds), average="macro")
        history.append({"epoch": epoch, "train_loss": round(running / len(loader), 4), "val_macro_f1": round(val_f1, 4)})
        print(history[-1])
        if val_f1 > best_f1:
            best_f1 = val_f1
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    return model, history, time.time() - t0

def count_params(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    return trainable, total

## Эксперимент 2. Full fine-tuning

In [6]:
FT = {"learning_rate": 5e-5, "batch_size": 16, "epochs": 6, "optimizer": "AdamW (weight_decay=0.01)",
      "loss": "weighted CrossEntropy", "scheduler": "linear warmup 10%", "max_len": MAX_LEN}
set_seed()
ft_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(INCIDENT_TYPES), id2label=id2label, label2id=label2id)
ft_trainable, ft_total = count_params(ft_model)
print(f"Обучаемых параметров: {ft_trainable:,} из {ft_total:,} (100%)")
ft_model, ft_history, ft_time = train(ft_model, FT["learning_rate"], FT["epochs"], FT["batch_size"])
results["Fine-tuning rubert-tiny2 (все веса)"] = {
    "val": metrics(val_df["type"], predict(ft_model, val_ds)),
    "test": metrics(test_df["type"], predict(ft_model, test_ds)),
    "trainable_params": ft_trainable, "total_params": ft_total,
    "train_time_sec": round(ft_time, 1), "params": FT, "history": ft_history,
}
print(classification_report(test_df["type"], predict(ft_model, test_ds), zero_division=0))

Loading weights: 100%|██████████| 55/55 [00:00<00:00, 4559.84it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the ch

Обучаемых параметров: 29,195,020 из 29,195,020 (100%)
{'epoch': 1, 'train_loss': 1.3686, 'val_macro_f1': 0.2827}
{'epoch': 2, 'train_loss': 0.995, 'val_macro_f1': 0.6846}
{'epoch': 3, 'train_loss': 0.6396, 'val_macro_f1': 0.7083}
{'epoch': 4, 'train_loss': 0.4681, 'val_macro_f1': 0.9099}
{'epoch': 5, 'train_loss': 0.3932, 'val_macro_f1': 0.9099}
{'epoch': 6, 'train_loss': 0.3629, 'val_macro_f1': 0.9099}
                   precision    recall  f1-score   support

           Авария       1.00      1.00      1.00         2
      Микротравма       0.96      0.89      0.92        62
Несчастный случай       1.00      0.89      0.94         9
 Опасная ситуация       0.96      0.99      0.97       151

         accuracy                           0.96       224
        macro avg       0.98      0.94      0.96       224
     weighted avg       0.96      0.96      0.96       224



## Эксперимент 3. LoRA (Low-Rank Adaptation)

**Идея.** Исходная матрица весов W (например, 312×312 в слое внимания) **замораживается**. Обучается только её поправка в виде произведения двух узких матриц:

  W' = W + (α / r) · B·A,  где A — r×312, B — 312×r, а r ≪ 312.

При r = 8 вместо 97 344 весов одной матрицы обучаются 2·8·312 = 4 992, то есть ≈ 5%.

| Параметр | Значение | Смысл |
|---|---|---|
| `r` (rank) | 8 | «ширина» адаптера; больше — гибче, но больше параметров |
| `lora_alpha` | 16 | масштаб поправки (α/r = 2) |
| `lora_dropout` | 0.1 | регуляризация адаптера |
| `target_modules` | `query`, `value` | матрицы внимания, где адаптация наиболее эффективна (как в статье LoRA) |
| классификатор | обучается полностью | новый слой под 4 класса (`modules_to_save`) |

LoRA учится медленнее, поэтому learning rate выше (5e-4), а эпох больше (10).

In [7]:
LORA = {"r": 8, "lora_alpha": 16, "lora_dropout": 0.1, "target_modules": ["query", "value"],
        "learning_rate": 5e-4, "batch_size": 16, "epochs": 10, "optimizer": "AdamW (weight_decay=0.01)",
        "loss": "weighted CrossEntropy"}
set_seed()
base = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(INCIDENT_TYPES), id2label=id2label, label2id=label2id)
lora_cfg = LoraConfig(task_type=TaskType.SEQ_CLS, r=LORA["r"], lora_alpha=LORA["lora_alpha"],
                      lora_dropout=LORA["lora_dropout"], target_modules=LORA["target_modules"])
lora_model = get_peft_model(base, lora_cfg)
lora_model.print_trainable_parameters()
lora_trainable, lora_total = count_params(lora_model)
lora_model, lora_history, lora_time = train(lora_model, LORA["learning_rate"], LORA["epochs"], LORA["batch_size"])
results["LoRA rubert-tiny2 (r=8, q/v)"] = {
    "val": metrics(val_df["type"], predict(lora_model, val_ds)),
    "test": metrics(test_df["type"], predict(lora_model, test_ds)),
    "trainable_params": lora_trainable, "total_params": lora_total,
    "train_time_sec": round(lora_time, 1), "params": LORA, "history": lora_history,
}
print(classification_report(test_df["type"], predict(lora_model, test_ds), zero_division=0))

Loading weights: 100%|██████████| 55/55 [00:00<00:00, 7039.79it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the ch

trainable params: 31,204 || all params: 29,226,224 || trainable%: 0.1068
{'epoch': 1, 'train_loss': 1.3788, 'val_macro_f1': 0.2645}
{'epoch': 2, 'train_loss': 1.177, 'val_macro_f1': 0.4259}
{'epoch': 3, 'train_loss': 0.8689, 'val_macro_f1': 0.4568}
{'epoch': 4, 'train_loss': 0.7387, 'val_macro_f1': 0.7055}
{'epoch': 5, 'train_loss': 0.6209, 'val_macro_f1': 0.6607}
{'epoch': 6, 'train_loss': 0.531, 'val_macro_f1': 0.7055}
{'epoch': 7, 'train_loss': 0.4435, 'val_macro_f1': 0.7083}
{'epoch': 8, 'train_loss': 0.3831, 'val_macro_f1': 0.8313}
{'epoch': 9, 'train_loss': 0.3762, 'val_macro_f1': 0.9071}
{'epoch': 10, 'train_loss': 0.3947, 'val_macro_f1': 0.9043}
                   precision    recall  f1-score   support

           Авария       1.00      0.50      0.67         2
      Микротравма       0.96      0.89      0.92        62
Несчастный случай       1.00      0.89      0.94         9
 Опасная ситуация       0.95      0.99      0.97       151

         accuracy                        

## Сводка и сохранение результатов

In [8]:
table = pd.DataFrame({name: {"val acc": r["val"]["accuracy"], "val macro-F1": r["val"]["macro_f1"],
                             "test acc": r["test"]["accuracy"], "test macro-F1": r["test"]["macro_f1"],
                             "обучаемых параметров": r["trainable_params"], "время, с": r["train_time_sec"]}
                      for name, r in results.items()}).T
table

,val acc,val macro-F1,test acc,test macro-F1,обучаемых параметров,"время, с"
Transfer learning (frozen rubert-tiny2 + LogReg),0.9283,0.8909,0.9107,0.8858,1252.0,41.2
Fine-tuning rubert-tiny2 (все веса),0.9596,0.9099,0.9598,0.9599,29195020.0,376.2
"LoRA rubert-tiny2 (r=8, q/v)",0.9552,0.9071,0.9554,0.8758,31204.0,513.6


In [9]:
out = ROOT / "reports" / "transformer_results.json"
out.parent.mkdir(exist_ok=True)
payload = {"model_name": MODEL_NAME, "device": str(device), "seed": SEED,
           "versions": {"torch": torch.__version__, "transformers": transformers.__version__,
                        "peft": peft.__version__, "sklearn": sklearn.__version__},
           "results": results}
out.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
print("Сохранено:", out)
if IN_COLAB:
    from google.colab import files
    files.download(str(out))   # положите файл в reports/ своего проекта

Сохранено: c:\Users\Admin\PY Projects\AITU\HSE_AI_Essentials\reports\transformer_results.json


## Интерпретация

**Почему PEFT (LoRA) предпочтительнее полного fine-tuning:**
1. **Параметры.** Обучается лишь несколько процентов весов. Нужно меньше памяти GPU, а вероятность переобучения на маленьком датасете (≈ 1 000 текстов в train) ниже.
2. **Хранение.** Адаптер весит единицы мегабайт вместо сотни. Для нескольких задач (тип, категория Коргау, причина) хранится одна базовая модель и несколько адаптеров.
3. **Сохранение знаний.** Исходные веса не меняются, поэтому модель не «забывает» язык (catastrophic forgetting).

**Чего ожидать на этих данных.** Тексты синтетические и частично шаблонные, поэтому сильная классическая модель (TF-IDF + LinearSVC) уже даёт macro-F1 ≈ 0.9. Трансформер выигрывает там, где важны синонимы и перефразирование, а это особенно ценно на реальных отчётах. Сравнение всех моделей на одном test — в `03_model_training.ipynb`.